# Phase 2.3 : Diagnostic comparatif TP1 vs TP2

Chats vs chiens (Microsoft Cats vs Dogs Dataset). Suite de `phase2_2_dropout.ipynb`.

**Important — ce notebook n'est PAS autonome comme les précédents** : il compare les résultats de deux entraînements déjà faits (phase 1.4 : CNN from scratch, phase 2.2 : CNN augmenté + Dropout). Il a besoin des fichiers `history_scratch.json`, `history_augmented.json`, `model_scratch.keras` et `model_augmented.keras`.

**Deux façons de les obtenir** :
1. **Même runtime Colab** (recommandé) : exécute `phase1_4_training.ipynb` puis `phase2_2_dropout.ipynb` dans ce même environnement d'exécution, sans redémarrer — les 4 fichiers seront déjà présents dans `/content/`.
2. **Runtime différent** : télécharge les 4 fichiers depuis la session où tu as exécuté les phases 1.4 et 2.2, puis uploade-les ici (`from google.colab import files; files.upload()`).

In [ ]:
import os

required = ["history_scratch.json", "history_augmented.json", "model_scratch.keras", "model_augmented.keras"]
missing = [f for f in required if not os.path.exists(f)]

if missing:
    print("Fichiers manquants :", missing)
    print("Execute phase1_4_training.ipynb et phase2_2_dropout.ipynb dans ce runtime avant de continuer,")
    print("ou uploade-les manuellement (cellule suivante).")
else:
    print("Tous les fichiers necessaires sont presents.")

In [ ]:
# A executer seulement si la cellule precedente signale des fichiers manquants.
from google.colab import files
files.upload()  # selectionner history_scratch.json, history_augmented.json, model_scratch.keras, model_augmented.keras

In [ ]:
import json
import tensorflow as tf

with open("history_scratch.json") as f:
    history_scratch_dict = json.load(f)
with open("history_augmented.json") as f:
    history_augmented_dict = json.load(f)

model_scratch = tf.keras.models.load_model("model_scratch.keras")
model_augmented = tf.keras.models.load_model("model_augmented.keras")

print("val_accuracy max (scratch)   :", f"{max(history_scratch_dict['val_accuracy']):.3f}")
print("val_accuracy max (augmente)  :", f"{max(history_augmented_dict['val_accuracy']):.3f}")

## Phase 2.3 : Diagnostic comparatif TP1 vs TP2

**Objectif** : comparer systématiquement les courbes des deux modèles, mettre les chiffres en regard, et rédiger un paragraphe d'interprétation.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history_scratch_dict["val_loss"], label="Scratch (TP1)", color="red")
ax1.plot(history_augmented_dict["val_loss"], label="Augmente + Dropout (TP2)", color="blue")
ax1.set_title("Val Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(history_scratch_dict["val_accuracy"], label="Scratch (TP1)", color="red")
ax2.plot(history_augmented_dict["val_accuracy"], label="Augmente + Dropout (TP2)", color="blue")
ax2.set_title("Val Accuracy")
ax2.set_xlabel("Epoch")
ax2.legend()

plt.tight_layout()
plt.savefig("comparison_tp1_tp2.png", dpi=100)
plt.show()

In [ ]:
val_acc_scratch = max(history_scratch_dict["val_accuracy"])
val_acc_augmented = max(history_augmented_dict["val_accuracy"])

gap_scratch = max(history_scratch_dict["accuracy"]) - val_acc_scratch
gap_augmented = max(history_augmented_dict["accuracy"]) - val_acc_augmented

print(f"{'Modele':<25} {'val_acc':>10} {'train-val gap':>15} {'epochs':>8} {'params':>12}")
print("-" * 75)
print(f"{'CNN scratch (TP1)':<25} {val_acc_scratch:>9.1%} {gap_scratch:>14.1%} "
      f"{len(history_scratch_dict['val_accuracy']):>8} {model_scratch.count_params():>12,}")
print(f"{'CNN augmente (TP2)':<25} {val_acc_augmented:>9.1%} {gap_augmented:>14.1%} "
      f"{len(history_augmented_dict['val_accuracy']):>8} {model_augmented.count_params():>12,}")

### À toi de rédiger (5-8 lignes, dans cette cellule)

Réponds à ces questions à partir des chiffres et courbes ci-dessus :

- Qu'est-ce qui a changé entre TP1 et TP2 dans les courbes ?
- Le gap train/val s'est-il réduit ? De combien ?
- La convergence est-elle plus lente ou plus rapide ? Pourquoi ?
- Qu'est-ce qui reste insuffisant pour ce dataset ? Pourquoi l'augmentation seule ne suffit pas (→ transfer learning en TP3) ?

*(remplace ce texte par ton interprétation)*

### Avant de passer à la phase 3.1

- **Happy path** : graphe de superposition sauvegardé, tableau de métriques affiché, TP2 montre `val_acc` > TP1 et un gap train/val plus faible.
- **Edge case** : la comparaison n'a de sens que si les deux modèles ont tourné sur le même nombre d'epochs effectif — si l'un s'est arrêté beaucoup plus tôt via `EarlyStopping`, regarde `len(history[...]['val_accuracy'])` avant de conclure.
- **Adversarial** : si tu relances les deux phases avec des seeds différentes (`tf.random.set_seed(0)`, `1`, `42`), l'écart entre seeds peut être plus grand que l'écart TP1 vs TP2 — dans ce cas, le gain observé serait dans le bruit statistique plutôt que dans la régularisation. Question honnête à se poser avant de conclure trop vite.

**Prochaine étape** : Phase 3.1 — transfer learning avec MobileNetV2.